# **East Africa Jobs Skills Demand Forecaster**

>


In [ ]:
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
df1= pd.read_excel('/content/drive/My Drive/Data Analysis Portfolio Projects/datasets/Becky/east_africa_jobs_part_01.xlsx')
df2= pd.read_excel('/content/drive/My Drive/Data Analysis Portfolio Projects/datasets/Becky/east_africa_jobs_part_02.xlsx')
df3= pd.read_excel('/content/drive/My Drive/Data Analysis Portfolio Projects/datasets/Becky/east_africa_jobs_part_03.xlsx')
df4= pd.read_excel('/content/drive/My Drive/Data Analysis Portfolio Projects/datasets/Becky/east_africa_jobs_part_04.xlsx')
df5= pd.read_excel('/content/drive/My Drive/Data Analysis Portfolio Projects/datasets/Becky/east_africa_jobs_part_05.xlsx')



In [ ]:
df = pd.concat([df1, df2, df3, df4, df5], ignore_index=True)

In [ ]:
df.head()

,Job_ID,Job_Title,Company,Company_Type,Location,Country,Industry,Salary_Range_USD,Description,Requirements,...,Job_Type,Posting_Date,Application_Deadline,Required_Education,Languages_Required,Benefits,Remote_Work_Option,Contact_Email,Application_Method,Sector
0,EA000001,Project Manager,Ministry of Construction,Government,Kitale,Kenya,Construction,9568-12945,Exciting opportunity at Ministry of Constructi...,Minimum 0-2 years of experience; Ability to wo...,...,Hybrid,2023-10-16,2023-11-20,Bachelor's,English,Comprehensive medical cover; Pension scheme; H...,No,careers@ministryofconstruction.com,Online Application,Public
1,EA000002,AI/ML Specialist,Unity Technology Initiative,NGO,Kigoma,Tanzania,Technology & IT,35837-48486,Join Unity Technology Initiative as a AI/ML Sp...,Minimum 0-2 years of experience; Attention to ...,...,Full-time,2024-08-28,2024-11-10,Diploma,Swahili,"Health Insurance, Pension",No,careers@unitytechnologyinitiative.com,Online Application,Non-Profit
2,EA000003,Tourism Marketing Manager,National Tourism Partners,Startup,Kisumu,Kenya,Tourism & Hospitality,11393-15414,Exciting opportunity at National Tourism Partn...,Minimum 0-2 years of experience; Ability to wo...,...,Remote,2024-02-20,2024-03-30,Master's,Swahili,"Health Insurance, Pension",No,careers@nationaltourismpartners.com,Online Application,Non-Profit
3,EA000004,Branch Manager,Elite Finance Holdings,Private Company,Mbarara,Uganda,Finance & Banking,27400-37071,Exciting opportunity at Elite Finance Holdings...,Minimum 0-2 years of experience; Strong commun...,...,Remote,2024-02-13,2024-05-04,Bachelor's,Swahili,"Health Insurance, Pension",No,careers@elitefinanceholdings.com,Online Application,Private
4,EA000005,Data Analyst,International Technology Agency,International Organization,Morogoro,Tanzania,Technology & IT,28875-39066,Join International Technology Agency as a Data...,Minimum 6-10 years of experience; Attention to...,...,Remote,2024-07-09,2024-08-26,Bachelor's,Multiple Languages,Comprehensive medical cover; Pension scheme; H...,Yes,careers@internationaltechnologyagency.com,Online Application,Non-Profit


In [ ]:
df.columns

Index(['Job_ID', 'Job_Title', 'Company', 'Company_Type', 'Location', 'Country',
       'Industry', 'Salary_Range_USD', 'Description', 'Requirements',
       'Experience_Level', 'Job_Type', 'Posting_Date', 'Application_Deadline',
       'Required_Education', 'Languages_Required', 'Benefits',
       'Remote_Work_Option', 'Contact_Email', 'Application_Method', 'Sector'],
      dtype='object')

In [ ]:
df.shape

(50000, 21)

## **Data Cleaning**

In [ ]:
df['Posting_Date'] = pd.to_datetime(df['Posting_Date'])
df['Application_Deadline'] = pd.to_datetime(df['Application_Deadline'])

In [ ]:
df.columns= df.columns.str.lower()

df.columns

Index(['job_id', 'job_title', 'company', 'company_type', 'location', 'country',
       'industry', 'salary_range_usd', 'description', 'requirements',
       'experience_level', 'job_type', 'posting_date', 'application_deadline',
       'required_education', 'languages_required', 'benefits',
       'remote_work_option', 'contact_email', 'application_method', 'sector'],
      dtype='object')

In [ ]:
df['job_title'].unique()

array(['Project Manager', 'AI/ML Specialist', 'Tourism Marketing Manager',
       'Branch Manager', 'Data Analyst', 'Transport Supervisor',
       'Mobile App Developer', 'Livelihoods Specialist',
       'Investment Advisor', 'Technical Support Engineer', 'Nutritionist',
       'Midwife', 'Medical Records Officer', 'Travel Consultant',
       'Financial Analyst', 'Soil Scientist', 'Treasury Analyst',
       'Program Manager', 'University Lecturer', 'Tour Guide',
       'Systems Analyst', 'Fleet Manager', 'Bank Teller',
       'Agricultural Economist', 'Education Officer',
       'Curriculum Developer', 'Auditor', 'Cybersecurity Analyst',
       'Hotel Receptionist', 'Maintenance Engineer',
       'Financial Controller', 'Social Media Manager', 'Health Educator',
       'Construction Safety Officer', 'Loan Officer',
       'Quality Control Supervisor', 'Urban Planner', 'Medical Doctor',
       'Agribusiness Manager', 'Grant Writer', 'Risk Manager',
       'Education Consultant', 'Archit

## **Skills Extraction**

### **1. Combine text columns**

In [ ]:
df['text'] = df['job_title'].fillna('')

### **2. NLP Preprocessing**

In [ ]:
import spacy

nlp = spacy.load("en_core_web_sm")  # small English model



In [ ]:
def clean_text(text):
    doc = nlp(str(text).lower())
    tokens = [token.lemma_ for token in doc if token.is_alpha and not token.is_stop]
    return " ".join(tokens)



In [ ]:
df['clean_text'] = df['text'].apply(clean_text)

### **3. Build a Skills Dictionary**

In [ ]:
jobs = df['job_title'].str.lower().unique().tolist()

### **4. Extract Jobs**


In [ ]:
from sentence_transformers import SentenceTransformer, util

# Load lightweight embedding model
model = SentenceTransformer('all-MiniLM-L6-v2')

# Encode jobs once
job_embeddings = model.encode(jobs, convert_to_tensor=True)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
import torch

def extract_jobs_bert(text, jobs=jobs, job_embeddings=job_embeddings, threshold=0.6):
    # Encode job text
    job_embedding = model.encode(text, convert_to_tensor=True)

    # Cosine similarity
    cos_scores = util.cos_sim(job_embedding, job_embeddings)

    # Return skills above threshold
    extracted = [jobs[i] for i in range(len(jobs)) if cos_scores[0][i] > threshold]
    return extracted

In [ ]:
df['jobs'] = df['clean_text'].apply(lambda x: extract_jobs_bert(x))

### **5. Explode jobs(per row)**

In [ ]:
jobs_df = df.explode('jobs')
jobs_df = jobs_df.dropna(subset=['jobs'])

### **6. Aggregate skill demand by month & country**

In [ ]:
jobs_df['posting_date'] = pd.to_datetime(jobs_df['posting_date'])
jobs_df['month'] = jobs_df['posting_date'].dt.to_period('M')

skill_demand = jobs_df.groupby(['jobs','country','month']).size().reset_index(name='demand')

## **Modelling**

### **Overview**

> In this section, we will build a forecasting model to predict the demand for specific IT skills in East Africa over the next 6–24 months. We will:

 >1. Aggregate historical job postings per skill and country.

>2. Apply BERT + dictionary mapping to extract skills from job descriptions.

>3. Train Prophet models for each skill × country combination.

>4. Save models for future predictions in our Flask application.


In [ ]:
base_path = '/content/drive/My Drive/Data Analysis Portfolio Projects/datasets/Becky/'

In [ ]:
df['jobs']

,jobs
0,"[project manager, program manager, it project ..."
1,[ai/ml specialist]
2,"[tourism marketing manager, hotel manager, mar..."
3,[branch manager]
4,"[data analyst, financial analyst, systems anal..."
...,...
49995,"[logistics manager, supply chain manager, supp..."
49996,"[architect, structural engineer]"
49997,[merchandiser]
49998,[nutritionist]


#### **1. Select Forecasting Model**

>Based on data characteristics (sparse time series, missing months, multiple skill × country series), we choose **Prophet**:

>* Handles missing months automatically

>* Captures trends & seasonality

>* Minimal parameter tuning

>* Scales easily across multiple series

#### **2. Train Prophet Models**

In [ ]:
from prophet import Prophet
import pickle

prophet_models = {}  # dictionary to store trained models

for skill in skill_demand['jobs'].unique():
    for country in skill_demand['country'].unique():
        # Filter series
        series = skill_demand[(skill_demand.jobs==skill) &
                              (skill_demand.country==country)][['month','demand']]
        if len(series) < 6:
            continue  # skip very short series

        # Prepare Prophet format
        df_prophet = series.rename(columns={'month':'ds','demand':'y'})
        df_prophet['ds'] = df_prophet['ds'].dt.to_timestamp()  # convert Period to Timestamp

        # Initialize & fit Prophet
        model = Prophet(yearly_seasonality=True, weekly_seasonality=False, daily_seasonality=False)
        model.fit(df_prophet)

        # Store model
        prophet_models[(skill, country)] = model

# Save all models for Flask
with open(base_path  + 'prophet_models.pkl','wb') as f:
    pickle.dump(prophet_models,f)

INFO:prophet:n_changepoints greater than number of observations. Using 11.
INFO:prophet:n_changepoints greater than number of observations. Using 11.
INFO:prophet:n_changepoints greater than number of observations. Using 11.
INFO:prophet:n_changepoints greater than number of observations. Using 11.
INFO:prophet:n_changepoints greater than number of observations. Using 8.
INFO:prophet:n_changepoints greater than number of observations. Using 11.
INFO:prophet:n_changepoints greater than number of observations. Using 11.
INFO:prophet:n_changepoints greater than number of observations. Using 11.
INFO:prophet:n_changepoints greater than number of observations. Using 11.
INFO:prophet:n_changepoints greater than number of observations. Using 11.
INFO:prophet:n_changepoints greater than number of observations. Using 11.
INFO:prophet:n_changepoints greater than number of observations. Using 10.
INFO:prophet:n_changepoints greater than number of observations. Using 11.
INFO:prophet:n_changepoint

## **Save Datasets & Model for use in Web App**

In [ ]:

df.columns = df.columns.str.lower()  # lowercase columns

# Convert dates
df['posting_date'] = pd.to_datetime(df['posting_date'], errors='coerce')
df['application_deadline'] = pd.to_datetime(df['application_deadline'], errors='coerce')


# Cleaned dataset
df.to_csv(base_path + 'cleaned_jobs_dataset.csv', index=False)

# Skills demand dataset
skill_demand.to_csv(base_path + 'job_demand.csv', index=False)


In [ ]:
df.head()

In [ ]:
# Cleaned dataset
df.to_csv(base_path + 'cleaned_jobs_dataset.csv', index=False)